# Agent memory: taking context beyond the window

> Previous lectures scaled the Agent step by step: lecture 08 had the Agent work back and forth over long documents and repeated retrieval, lecture 09's post-training taught the model to call tools, and lecture 10 sent the Agent into a large codebase to fix bugs over many turns. They all hit the same bound: the context window cannot hold that much information.
>
> This lecture faces that bound directly, and discusses how an Agent with a finite window can have near-unlimited memory. We answer along three lines: MemGPT treats context as scarce memory and lets the model decide what to swap out and what to bring back; Cartridges do not shuttle text, but compress an entire corpus into a set of trainable vectors; CacheBlend, on the engineering side, stores already-computed intermediate results and reuses them.

How many turns an 8k window can hold can be computed by hand.

The model's context window is 8000 tokens. After subtracting about 1000 tokens of system instructions, 7000 tokens remain for the dialogue.

Each turn (user message, assistant reply, retrieval result) consumes about 700 tokens.

So the window fills in about 10 turns: $7000 \div 700 = 10$. Expanding the window to 128k only pushes this number to about 180 turns; it does not remove the bound.

Information that does not fit in the window does not exist for the model. This is not a failure of intelligence; it is the model's fixed design: at inference time the model can only see tokens inside the window, and it stores nothing on its own. To break this bound, the organization of information has to change: evict the oldest messages from the main context and retrieve them when needed, compress an entire corpus offline into trainable vectors, or cache already-computed intermediate results and reuse them. These three routes are the **memory** we add to the Agent.

This lecture starts from the most concrete fact: how many turns an Agent's window can hold.

## 1. Why memory matters

This section first settles one concrete fact: how many turns an Agent's window can hold. In a real system, an engineer has to estimate how long a dialogue can last before deciding when to bring in a memory mechanism. We first meet the smallest unit for measuring text, then estimate the cost of a dialogue, and finally look at where memory is strong and where it is weak inside the window.

The model does not count text by "characters". It counts a smaller unit called a token. In English, one token is about three-quarters of a word; in Chinese, one character often takes one or two tokens. The context window draws a hard bound for every Agent: at inference time the model can only see tokens inside the window; everything outside does not exist.

Assigning an order of magnitude to each part of a dialogue is enough to estimate how many turns it can last:

| Component | Magnitude per turn (tokens) |
|:---|:---|
| System instructions (fixed) | about 1000 |
| User message | about 200 |
| Assistant reply | about 300 |
| Retrieval result | about 200 |

For an 8k window, about 7000 tokens remain after the fixed overhead, each turn costs about 700 tokens, and the window fills in about ten turns. Expanding the window to 128k only postpones that moment; it does not remove it.

Memory strength is also uneven inside the window. Lost in the Middle shows empirically that the model remembers tokens at both ends of the context more firmly, and tends to ignore the middle. Filling the window is not the same as remembering all of the information. To break the window's limit, the organization of information has to change: eviction, compression, reuse. We start from the most concrete window budget, and see at which turn the dialogue fills the window.

In [ ]:
# Hand calculation: how many dialogue turns an 8k-window Agent can hold
window = 8000
system = 1000               # fixed cost of system instructions
per_turn = 700              # user 200 + assistant 300 + retrieval 200
usable = window - system
turns = usable // per_turn
warn = int(0.7 * window)
warn_turn = (warn - system) // per_turn + 1
print(f"usable dialogue budget = {window} - {system} = {usable} tokens")
print(f"each turn costs {per_turn} tokens; the window fills after about {turns} turns")
print(f"the 70% warning line at {warn} tokens appears near turn {warn_turn}")
print(f"Key observation: {turns} turns already fill the window, and long-horizon tasks far exceed that")


In [ ]:
# Visualization: main-context occupancy grows linearly with turns
import matplotlib.pyplot as plt
import numpy as np

window = 8000
warn = int(0.7 * window)
per_turn = 700
system = 1000
turn_ids = np.arange(0, 16)
occupancy = system + per_turn * turn_ids

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(turn_ids, occupancy, marker="o", label="main context")
ax.axhline(window, color="r", linestyle="--", label="window limit (100%)")
ax.axhline(warn, color="orange", linestyle="--", label="warning line (70%)")
ax.set_xlabel("turn")
ax.set_ylabel("tokens")
ax.set_title("Context occupancy vs dialogue turns")
ax.legend()
plt.tight_layout()
plt.show()

first_warn = np.flatnonzero(occupancy >= warn)[0]
first_full = np.flatnonzero(occupancy >= window)[0]
print(f"turn {first_warn} crosses the warning line; turn {first_full} hits the limit")


### Why the window fills: turn-by-turn hand calculation

This subsection computes the previous estimate step by step, to see how window occupancy grows turn by turn and why it is full by turn 10.

The context window is the total number of tokens the model can see in one inference pass; content outside the window does not exist for it. Estimating dialogue scale does not need to be exact. We adopt the following convention: a user message is about 200 tokens, an assistant reply about 300 tokens, and a retrieval result about 200 tokens, for about 700 tokens per turn.

The key to occupancy growth is accumulation. By default, the prompt the model sees each turn contains every previous turn's messages: user messages, assistant replies, and retrieval results are all kept as-is, so occupancy at turn t is the sum of all previous turns. Occupancy therefore grows linearly with the number of turns. For an 8k window (8000 tokens), turn by turn:

| Turn | Window occupancy (tokens) | Note |
|:---|:---|:---|
| 0 | 1000 | system instructions only |
| 1 | 1700 | +700 |
| 2 | 2400 | +700 |
| 3 | 3100 | +700 |
| 4 | 3800 | +700 |
| 5 | 4500 | +700 |
| 6 | 5200 | +700 |
| 7 | 5900 | crosses the 70% warning line (5600) |
| 8 | 6600 | +700 |
| 9 | 7300 | +700 |
| 10 | 8000 | window full |

The arithmetic: after subtracting the fixed system instructions, the usable budget is $8000 - 1000 = 7000$ tokens; each turn adds 700 tokens, so the window lasts $7000 \div 700 = 10$ turns. The 70% warning line is at $0.7 \times 8000 = 5600$ tokens; after subtracting system instructions, 4600 tokens remain, $4600 \div 700 \approx 6.6$, so turn 7 crosses the warning line. Stretching the window to 128k only postpones the same arithmetic. Occupancy at turn t is still $1000 + 700t$; filling the window is only a matter of time.

What to do after the window is full is what this lecture answers at three levels. The most direct route is to evict the oldest messages from the main context and retrieve them when needed: MemGPT's scheduling line. We can also skip shuttling text and compress an entire corpus offline into a set of trainable vectors that the model reads at query time: Cartridges' compression line. We can also cache computed intermediate results and skip recomputation when the same content reappears: CacheBlend's reuse line.

Occupancy keeps growing, so the main context must stay bounded. The information itself does not disappear, but when it no longer fits, something has to decide which message gives up its place. That decision is called eviction. Operating systems, database caches, and browser tabs all use eviction rules to decide what to drop first; an Agent's external memory is the same.

The three eviction policies each follow one criterion. FIFO follows arrival order: the first in is the first out. Importance scoring follows semantic value: the least important leaves first. LRU follows recency of use: the least recently used leaves first. On the same set of messages, the three policies pick different victims. We compare them on a toy dataset.

In [ ]:
# Three memories to manage: arrival order, importance, last-used time
memories = [
    {"id": "a", "ts": 0, "importance": 0.3, "last_used": 5},
    {"id": "b", "ts": 1, "importance": 0.9, "last_used": 8},
    {"id": "c", "ts": 2, "importance": 0.6, "last_used": 2},
    {"id": "d", "ts": 3, "importance": 0.2, "last_used": 0},
]


def evict_fifo(memories):
    """FIFO: evict the earliest arrival (smallest ts); return its index."""
    return min(range(len(memories)), key=lambda i: memories[i]["ts"])


def evict_lowest_importance(memories):
    """Importance: evict the lowest-scoring item; return its index."""
    return min(range(len(memories)), key=lambda i: memories[i]["importance"])


def evict_lru(memories):
    """LRU: evict the least recently used item; return its index."""
    return min(range(len(memories)), key=lambda i: memories[i]["last_used"])


for name, fn in [("FIFO", evict_fifo),
                 ("importance", evict_lowest_importance),
                 ("LRU", evict_lru)]:
    idx = fn(memories)
    print(f"{name:10s} evicts {memories[idx]['id']}")
print("Key observation: on the same memories, the three policies pick different victims")


### Which item each eviction policy picks

This subsection reads the toy data item by item, to see whom each policy lets go first, and why.

The four memories above each have three fields: ts is arrival order, importance is a semantic-importance score, and last_used is the most recent time the item was used. Listing all four:

| id | ts (arrival order) | importance | last_used (recency) |
|:---|:---|:---|:---|
| a | 0 | 0.3 | 5 |
| b | 1 | 0.9 | 8 |
| c | 2 | 0.6 | 2 |
| d | 3 | 0.2 | 0 |

FIFO yields by arrival order: the smallest ts is a (ts=0), so a is evicted. It encodes the most basic intuition: the earliest message is already stale. The downside is that important old information can leave with it. If an address the user left three months ago sits in the oldest message, FIFO will yield it first.

The importance policy yields by semantic value: the lowest importance is d (0.2), so d is evicted. It needs a scoring mechanism to judge whether each memory is worth keeping; in MemGPT that score is given by the model itself. The downside is that scoring can be wrong, and the judgment ignores frequency of use.

LRU yields by recency: the smallest last_used is d (0), so d is evicted. It uses the regularity that a recently used memory is likely to be used again soon, which fits dialogue and other settings with strong temporal locality. The cost is that it cannot tell a high-frequency, high-value old memory from a low-frequency ordinary one.

On the same memories, the three policies pick different victims: FIFO picks a; importance and LRU both pick d. No policy is best in every setting. Real systems often combine them: first stratify by importance, then apply LRU inside each stratum.

In [ ]:
# Visualization: how memory contents change over turns under the three policies
import matplotlib.pyplot as plt
import numpy as np


def simulate(evict_fn, seq, capacity):
    """Insert seq in order; when over capacity, evict with evict_fn. Return the kept id set each turn."""
    kept = []
    store = []
    for item in seq:
        store.append(item)
        while len(store) > capacity:
            store.pop(evict_fn(store))
        kept.append(sorted(m["id"] for m in store))
    return kept


rng = np.random.default_rng(0)
seq = [{"id": f"m{i}", "ts": i,
        "importance": round(rng.uniform(0.1, 1), 2),
        "last_used": int(rng.integers(0, 12))} for i in range(12)]
capacity = 4

policies = {"FIFO": evict_fifo,
            "importance": evict_lowest_importance,
            "LRU": evict_lru}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, fn) in zip(axes, policies.items()):
    kept = simulate(fn, seq, capacity)
    grid = np.zeros((len(seq), len(kept)))
    for t, ids in enumerate(kept):
        for item_id in ids:
            grid[int(item_id[1:]), t] = 1
    ax.imshow(grid, aspect="auto", cmap="Blues", interpolation="nearest")
    ax.set_title(name)
    ax.set_xlabel("turn")
    ax.set_ylabel("memory id")
plt.tight_layout()
plt.show()

print("Each cell shows whether a memory is still in the main context that turn; the three policies keep different shapes.")


## 2. Treating context as finite memory

This section answers one question: once the window is full, where does information go, and how is it brought back when needed. In real systems, a customer-support assistant stores user preferences in long-term memory, and a programmer assistant stores frequently used codebase knowledge for use across sessions. Both follow the idea "move out what does not fit, fetch it back when needed". In this section we implement such a two-level store by hand.

As noted earlier, our own memory splits into short-term and long-term: what we are thinking about now is short-term memory; what has been stored for a long time and can be recalled at any moment is long-term memory. An Agent needs both. The context window is only the analogue of short-term memory; it cannot hold an entire project. Outside the window there has to be long-term memory, so that knowledge can persist across sessions. MemGPT borrows an operating-system arrangement: besides physical memory there is a disk; pages that are not needed for now are swapped to disk and swapped back when needed. Mapped onto the model, the context window is memory, external storage is disk, and the model itself decides what to swap out and what to swap in.

MemGPT splits the main context into three parts in a fixed order. System instructions are read-only and spell out control flow and function usage; working context is a read-write fixed-length text that holds key user facts and preferences; the FIFO queue is rolling message history that stores the most recent turns. Outside the main context is the external context, with two stores: recall storage holds every historical message (never deleted), and archival storage holds records that must be kept for the long term. Information swapped out of the main context enters the external context, and is fetched back when the model issues a function call. The data structures below turn this two-level store into code.

In [ ]:
# Two-level memory store: main context and external context
class MainContext:
    """Main context: system instructions + working context + FIFO message queue."""

    def __init__(self, system_text):
        self.system = system_text
        self.working = ""
        self.queue = []

    def tokens(self):
        """Estimate token occupancy; here character count is used as a proxy."""
        return (len(self.system) + len(self.working)
                + sum(len(m["content"]) for m in self.queue))

    def render(self):
        """Concatenate into prompt text in a fixed order."""
        parts = [self.system, "[working context]" + self.working]
        parts += [f"{m['role']}: {m['content']}" for m in self.queue]
        return "\n".join(parts)


class ExternalContext:
    """External context: recall store (full history) and archival store (long-term records)."""

    def __init__(self):
        self.recall = []
        self.archival = {}

    def log(self, role, content):
        """Write a message permanently into the recall store."""
        self.recall.append({"role": role, "content": content})

    def search(self, keyword):
        """Keyword search in the recall store; return up to the 3 most recent hits."""
        hits = [m for m in self.recall if keyword in m["content"]]
        return hits[-3:]


main = MainContext("You are a memory assistant. Keep memory bounded.")
ext = ExternalContext()
main.working = "user: Ming, fruit: apple"
main.queue.append({"role": "user", "content": "Nice weather today"})
ext.log("user", "My name is Ming, my favorite fruit is apple")
print("main-context token count:", main.tokens())
print("hits for 'apple':", len(ext.search("apple")), "item(s)")
print("main-context preview:", main.render()[:44] + "...")


### How main context and external context divide the work

This subsection spells out the two-level store piece by piece: which part is fast, which part is large, and how information moves between the two stores.

MainContext and ExternalContext above are the code form of the two-level store. The main context corresponds to physical memory in an operating system: fast, directly visible to the model, and limited in space. The external context corresponds to disk: large in capacity, but the model cannot see its contents directly and must fetch them on purpose. The code uses character count as a proxy for token count; the tokens method returns that proxy.

Inside the main context, three segments are laid out in a fixed order. System is read-only instructions that state the model's duties and available functions; working context is a fixed-length read-write text that holds key user facts and preferences, for example "user: Ming, fruit: apple"; the FIFO queue is rolling message history that stores only the most recent turns. The three segments concatenated in order are the prompt fed to the model.

The external context has two stores. Recall storage records every historical message; items only enter and never leave, so no message is truly lost. Archival storage holds records that must be kept for the long term, organized by keyword. Swap-out moves a message from the main context into the external context without deleting it: the message only changes from "visible to the model" to "retrievable when needed".

Recall is the inverse of swap-out. After receiving a question, the model first calls a search function to find related messages in the recall store, puts the hits back into the main context, and then generates an answer. The code above walks a full round: searching for "apple" hits "My name is Ming, my favorite fruit is apple" in the recall store and returns it to the model.

The point of the two-level structure is that swap-out solves capacity and recall solves availability. Messages are never lost; the cost is that the model has to know when to fetch, and with which keyword. That is why MemGPT lets the model itself decide how to read and write memory: the scheduling logic lives in the model's function calls, not hard-coded in an external program.

One issue remains: how the model itself decides which memory to write and which memory to look up. The method is to let the model read and write memory through function calls. Each function comes with a description that states its name, the parameters to pass, and what it does. A function executor parses function calls in the model's output, validates the parameters, runs them, and feeds the result (success information or an error) back to the model. Below we implement the four most common functions: append to working context, replace working context, write to archival, and search the recall store.

In [ ]:
import re


def parse_functions(text):
    """Parse a list of (name, args) from the model reply, of the form name(\"args\")."""
    return re.findall(r"(\w+(?:\.\w+)?)\s*\(\s*\"([^\"]*)\"\s*\)", text)


def execute_function(main, ext, name, args):
    """Run one memory function, update the two-level store, and return a result string."""
    if name == "working_context.append":
        main.working = (main.working + "\n" + args).strip()
        return "appended: " + args
    if name == "working_context.replace":
        old, new = args.split("->", 1)
        main.working = main.working.replace(old.strip(), new.strip())
        return f"rewrote: {old.strip()} -> {new.strip()}"
    if name == "archival.insert":
        key = args.split(":", 1)[0].strip()
        ext.archival[key] = args
        return "wrote archive: " + key
    if name == "recall.search":
        hits = ext.search(args)
        return "; ".join(m["content"] for m in hits) if hits else "not found"
    return "unknown function: " + name


reply = ('Thought: the user stated a new fact.\n'
         'Action: working_context.append("Ming likes basketball")\n'
         'Action: recall.search("apple")')
for name, args in parse_functions(reply):
    print("run:", name, "|", execute_function(main, ext, name, args))
print("working context now:", main.working)


Putting the components above together into a minimal MemGPT-style loop, we call it ToyAgent. Each turn ToyAgent receives a message and writes it into the main context and the recall store; maintain keeps the budget, and when over the limit it evicts the oldest message and merges the old summary with that message into a new summary, called a recursive summary; ask searches external storage by keyword, puts hits back into the main context, and then calls the model. We first initialize the LLM client, then define this agent.

In [ ]:
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()
print("LLM client:", client.model)


class ToyAgent:
    """A minimal MemGPT-style agent: two-level storage + functions + budget maintenance.

    main: MainContext; ext: ExternalContext. Each turn ingest one message,
    then maintain keeps main-context tokens within budget.
    """

    def __init__(self, client, system, budget, warn_ratio=0.7):
        self.client = client
        self.main = MainContext(system)
        self.ext = ExternalContext()
        self.budget = budget
        self.warn = int(budget * warn_ratio)
        self.summary = "(empty summary)"
        self.history_tokens = []

    def ingest(self, role, content):
        """Write the message into the main-context queue and the external recall store."""
        self.main.queue.append({"role": role, "content": content})
        self.ext.log(role, content)

    def maintain(self):
        """Maintain the budget: when over the limit, evict the oldest message and build a recursive summary until back within budget."""
        events = []
        while self.main.tokens() > self.budget:
            if self.main.tokens() >= self.warn:
                events.append("memory pressure warning")
            evicted = self.main.queue.pop(0)
            self.ext.log("system", "evicted: " + evicted["content"])
            self.summary = self._summarize(self.summary, evicted["content"])
            events.append("evicted: " + evicted["content"][:18] + "...")
        self.history_tokens.append(self.main.tokens())
        return events

    def _summarize(self, old, evicted):
        """Build a recursive summary with the LLM; the live-API demo falls back to deterministic concatenation."""
        if False:
            return old + " ~ " + evicted
        prompt = f"Existing summary: {old}\nNewly evicted message: {evicted}\nMerge into a new summary:"
        return self.client.chat([{"role": "user", "content": prompt}])

    def ask(self, question, keyword):
        """Search external memory, put hits back into the main context, then call the model."""
        hits = self.ext.search(keyword)
        context = "; ".join(m["content"] for m in hits) or "(no related memory)"
        self.main.queue.append({"role": "system", "content": "retrieved memory: " + context})
        reply = self.client.chat(
            [{"role": "user", "content": question + "\nMemory for reference: " + context}])
        self.main.queue.append({"role": "assistant", "content": reply})
        return reply, hits


In [ ]:
# Run a scripted dialogue: the budget is small, so eviction happens quickly
system = "You are a memory helper. Write key facts into working context."
agent = ToyAgent(client, system, budget=70)

script = [
    ("user", "I am Ming. Favorite fruit: apple. I live in Beijing."),
    ("user", "Nice weather today, good for a walk."),
    ("user", "The project is due next Friday; remind me."),
    ("user", "What else is on this week's schedule?"),
]
for role, content in script:
    agent.ingest(role, content)
    events = agent.maintain()
    print(f"[{role}] {content[:22]}")
    for e in events:
        print("   ", e)
    print(f"    main tokens = {agent.main.tokens()} / budget {agent.budget}")
print("Key observation: user facts are evicted from the main context across turns and enter external storage.")


In [ ]:
# Swap out then recall: the fact has been evicted; ask retrieves it from external storage and puts it back into the main context
before = "apple" in agent.main.render()
reply, hits = agent.ask("What is my favorite fruit?", "fruit")
after = "apple" in agent.main.render()
print("before the question, 'apple' in main context:", before)
print("retrieval hits:", [h["content"][:24] for h in hits])
print("after retrieval, 'apple' in main context:", after)
print("model reply:", reply, "(the live-API demo is placeholder output)")
print("Key observation: an evicted fact is recovered via recall.search and re-enters the main context.")


### Maintaining the budget and recursive summaries

This subsection looks closely at the two methods maintain and ask: why eviction uses while rather than if, why the summary is called recursive, and how evicted information is recovered.

ToyAgent's maintain holds the budget as a line: as long as the main-context token count exceeds budget, it evicts the oldest message in the queue and does two things. First, it records the evicted message in the recall store, marked "evicted", so the message has not truly disappeared. Second, it merges the old summary with the evicted message into a new summary.

The summary has a recursive structure. The existing summary compresses every previously evicted message; when a new item is evicted, old and evicted are merged into a new summary. Recursive here means each turn's summary is an incremental update on the previous turn, rather than recompressing the full history. In the code, the `if False` branch simulates the merge by string concatenation; in a real system this step is one LLM call.

maintain uses while rather than if, because evicting one item may not be enough. If occupancy is still above budget after an eviction, it continues to evict and to update the summary until occupancy falls back within budget. In this dialogue the system string is about 62 characters and the four messages are about 36 to 52 characters each; from the first message onward, occupancy each turn exceeds the budget of 70, so almost every turn triggers eviction.

ask demonstrates recall. The user asks "What is my favorite fruit?", the agent searches the recall store with the keyword "fruit", and hits two items: the original message written at the start, and the copy written at eviction, both "I am Ming. Favorite fruit: apple. I live in Beijing.". After the hits are put back into the main context, the model answers. Before the question, "apple" was already absent from the main context; after retrieval it reappears.

An evicted fact recovered via recall.search re-enters the main context. That is the core value of two-level storage: a capacity limit does not cause information loss; it only raises the cost of retrieval.

In [ ]:
# Visualization: main-context occupancy stays within budget
import matplotlib.pyplot as plt
import numpy as np

turn_ids = np.arange(len(agent.history_tokens))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(turn_ids, agent.history_tokens, marker="o", label="main context")
ax.axhline(agent.budget, color="r", linestyle="--", label="budget")
ax.axhline(agent.warn, color="orange", linestyle="--", label="warning")
ax.set_xlabel("turn")
ax.set_ylabel("tokens")
ax.set_title("MemGPT-style budget maintenance")
ax.legend()
plt.tight_layout()
plt.show()
print(f"every turn, the main context stays within the budget of {agent.budget}.")


## 3. Compressing old information into a reusable representation

The previous section's MemGPT shuttles text: every swap-in and swap-out makes the model reread the text, which costs inference. This section takes another route: instead of moving text, compress the entire corpus in advance into a set of adjustable vectors, and have the model read those vectors when it needs them. In real systems, compressing the knowledge of a book or a codebase into a representation that can be queried repeatedly is this kind of method.

A piece of knowledge can be represented as a sequence of numbers, called a vector. When the model reads text, it eventually turns each word into a set of vectors as well. If a set of trainable vectors is prepended to the model's input, the model can read them through attention and take knowledge from them. Attention picks vectors to read by similarity; this practice of "finding the most related content among a pile of vectors by similarity" is called vector retrieval, and it is used in both RAG and memory systems.

The vectors prepended to the input have no corresponding text; that method is called prefix-tuning. The set of vectors trained for one corpus is called a cartridge. One cartridge is the knowledge of that corpus, compressed in advance, so that any number of later questions can reuse the same copy. Below we first make the KV cache behind "the model reading vectors" precise, then implement and compare two training objectives.

### KV cache: the carrier of memory

This subsection fills in a prerequisite that Cartridges need: how the model "reads" a piece of information. Once that is clear, it is clear why compressing knowledge into vectors lets the model remember it.

When the model reads text, it turns each word into a set of numeric vectors, split by role into three kinds: query, key, and value. When the model answers a question, it takes its own query and compares similarity with each word's key; the higher the similarity, the more it attends to that word's value; finally it takes a similarity-weighted sum of all values to obtain this step's output. The keys and values of the whole passage need to be computed only once and can then be stored; every later new word reuses them and does not recompute. This stored intermediate result is the KV cache; computing it in advance before generation is called prefill.

The KV cache matters for memory because: once a piece of information is encoded as keys and values, the model's query can read it through attention. Text tokens are one encoding of memory; numeric vectors in the KV cache are another. Cartridges take the latter route: they do not move text, only already-compressed vectors.

Prefix-tuning pushes this idea one step further. In ordinary use, the input starts with text tokens, and the model computes keys and values for them. If the start is replaced by p trainable vectors with no corresponding text, the model still reads them through attention. These p vectors are virtual tokens; the count p is usually far smaller than the corpus token count.

A cartridge is the set of virtual vectors trained for one corpus. In the code, each layer maintains a set of vectors of shape [p, d]; the real model weights are all frozen, and only these vectors are trained. For a 27-token toy corpus, each layer is given only 8 virtual tokens. Training is done offline, and the cost is amortized over repeated queries: the corpus is trained once, and any number of later questions reuse the same cartridge.

The training objective is critical. Direct next-token prediction teaches the model "what the next word is", so it can only recite the corpus and cannot answer questions. Context-distillation changes the objective, so the model learns "how to answer questions when the corpus is in context". Below we compute this objective by hand on two small distributions.

In [ ]:
# Hand calculation: the KL objective of context-distillation
import torch


def d_kl(p, q):
    """KL divergence KL(P || Q) = sum p log(p / q), where p and q are probability distributions."""
    return (p * torch.log(p / q)).sum().item()


teacher = torch.tensor([0.02, 0.02, 0.90, 0.03, 0.03])    # corpus in context
student_before = torch.tensor([0.20, 0.20, 0.20, 0.20, 0.20])  # initial student
student_after = torch.tensor([0.02, 0.02, 0.90, 0.03, 0.03])

kl_before = d_kl(teacher, student_before)
kl_after = d_kl(teacher, student_after)
print(f"KL(teacher || initial student) = {kl_before:.3f}")
print(f"KL(teacher || distilled student) = {kl_after:.3f}")
print("Key observation: distillation pulls the student distribution near the teacher, and KL falls close to 0.")


### KL divergence: a distance between distributions

This subsection explains the loss in the previous cell: what it computes, and why it is computed that way.

The loss of context-distillation is KL divergence, which measures how close two probability distributions are. The definition is $KL(P \| Q) = \sum_i p_i \log \frac{p_i}{q_i}$: take the log ratio of p to q on each class, then take a p-weighted sum. log(p/q) can be read as the surprise of using q in place of p; KL is the weighted average of that surprise.

In the code above, the teacher distribution puts weight 0.90 on the correct answer; the initial student is uniform, 0.20 on each class. Computing term by term:

| Class | p (teacher) | q (initial student) | p·ln(p/q) |
|:---|:---|:---|:---|
| answer | 0.90 | 0.20 | 0.90 × ln 4.5 ≈ 1.354 |
| noise 1 | 0.02 | 0.20 | 0.02 × ln 0.1 ≈ −0.046 |
| noise 2 | 0.02 | 0.20 | 0.02 × ln 0.1 ≈ −0.046 |
| noise 3 | 0.03 | 0.20 | 0.03 × ln 0.15 ≈ −0.057 |
| noise 4 | 0.03 | 0.20 | 0.03 × ln 0.15 ≈ −0.057 |

Adding the five terms, $1.354 - 0.046 - 0.046 - 0.057 - 0.057 \approx 1.148$. The answer class contributes the largest term: the teacher thinks it should have 0.90, the student only gives 0.20, and that underestimate is amplified in the weighted sum. The noise classes also fail to match, but p itself is small, so the weight is low and the negative values are held down. KL is always nonnegative, and is 0 only when P and Q are identical; after distillation the student matches the teacher and KL falls to 0.

KL is asymmetric, $KL(P\|Q) \ne KL(Q\|P)$; strictly speaking it is not a distance. Here we always write KL(teacher‖student), because the teacher is the target distribution and the student is the side being trained; the direction cannot be reversed.

In the toy training the teacher gives the answer 0.85; the hand-calculation example above uses 0.90. Both are merely different choices of soft label. The real reason appears in a live system: the teacher is the model's softmax output after the corpus is placed in context, itself a soft distribution that does not put all probability on one word. The toy uses a closed form to simulate that soft label: the answer takes 0.85, and the remaining probability is spread over all classes. A hard label (1.0) would push the student toward overconfidence and would also lose the detail of the teacher distribution.

In [ ]:
# From scratch: a tiny causal language model, with prefix KV and trainable virtual keys/values
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(8)   # cap the thread count to avoid jitter on small tensors


class TinyGPT(nn.Module):
    """A tiny causal language model that can prepend prefix KV to the key/value sequence.

    When prefix_ids is given, real corpus tokens are used as the prefix (teacher);
    otherwise each layer's trainable virtual key/value vectors are used (student, i.e. the cartridge).
    Real weights are all frozen.
    """

    def __init__(self, vocab, d=48, n_head=4, n_layer=2, p=8):
        super().__init__()
        self.d, self.n_head, self.n_layer, self.p = d, n_head, n_layer, p
        self.embed = nn.Embedding(vocab, d)
        self.vk = nn.ParameterList(
            [nn.Parameter(torch.randn(p, d) * 0.02) for _ in range(n_layer)])
        self.vv = nn.ParameterList(
            [nn.Parameter(torch.randn(p, d) * 0.02) for _ in range(n_layer)])
        self.layers = nn.ModuleList([self._block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab, bias=False)

    def _block(self):
        """One layer: qkv projection, output projection, two-layer FFN, two LayerNorms."""
        return nn.ModuleDict({
            "qkv": nn.Linear(self.d, 3 * self.d),
            "out": nn.Linear(self.d, self.d),
            "ffn": nn.Sequential(nn.Linear(self.d, 2 * self.d),
                                 nn.GELU(),
                                 nn.Linear(2 * self.d, self.d)),
            "ln1": nn.LayerNorm(self.d),
            "ln2": nn.LayerNorm(self.d),
        })

    def _attention(self, q, k, v, n_prefix):
        """Causal multi-head attention. q has shape [B,T,d]; k/v have shape [B,T+n_prefix,d]."""
        B, T, d = q.shape
        h, dh = self.n_head, d // self.n_head
        q = q.view(B, T, h, dh).transpose(1, 2)
        k = k.view(B, -1, h, dh).transpose(1, 2)
        v = v.view(B, -1, h, dh).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (dh ** 0.5)
        mask = torch.triu(
            torch.ones(T, n_prefix + T, dtype=torch.bool),
            diagonal=n_prefix + 1)
        scores = scores.masked_fill(mask[None, None], float("-inf"))
        w = F.softmax(scores, dim=-1)
        return (w @ v).transpose(1, 2).reshape(B, T, d)

    def forward(self, ids, prefix_ids=None, return_all=False):
        """ids has shape [B,T]. When return_all, return [B,T,vocab]; otherwise return
        logits at the last position [B,vocab]."""
        B, T = ids.shape
        x = self.embed(ids)
        for i, blk in enumerate(self.layers):
            xn = blk["ln1"](x)
            qkv = blk["qkv"](xn)
            q, k, v = qkv.chunk(3, dim=-1)
            if prefix_ids is not None:
                pn = blk["ln1"](self.embed(prefix_ids))
                pkv = blk["qkv"](pn)
                pk, pv = pkv[:, :, self.d:2 * self.d], pkv[:, :, 2 * self.d:]
            else:
                pk = self.vk[i].unsqueeze(0).expand(B, -1, -1)
                pv = self.vv[i].unsqueeze(0).expand(B, -1, -1)
            k = torch.cat([pk, k], dim=1)
            v = torch.cat([pv, v], dim=1)
            x = x + blk["out"](self._attention(q, k, v, pk.size(1)))
            x = x + blk["ffn"](blk["ln2"](x))
        logits = self.lm_head(self.ln_f(x))
        return logits if return_all else logits[:, -1, :]


model = TinyGPT(vocab=20)
print("trainable parameter count:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("of which virtual key/value parameters:", sum(p.numel() for p in list(model.vk.parameters())
                                  + list(model.vv.parameters())))


In [ ]:
# Toy corpus and questions: the corpus is a character-profile dictionary; the questions are profile lookups
corpus_tokens = ["name", "ALICE", ".", "fruit", "apple", ".", "city",
                 "Beijing", ".", "name", "BOB", ".", "fruit", "banana",
                 ".", "city", "Shanghai", ".", "name", "EVE", ".", "fruit",
                 "grape", ".", "city", "Paris", "."]
questions = [(("ALICE", "fruit"), "apple"),
             (("BOB", "fruit"), "banana"),
             (("ALICE", "city"), "Beijing"),
             (("BOB", "city"), "Shanghai")]
vocab = sorted(set(corpus_tokens)
               | set(a for _, a in questions)
               | set(t for q, _ in questions for t in q))
stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
corpus_ids = torch.tensor([stoi[w] for w in corpus_tokens]).unsqueeze(0)
print("vocabulary:", vocab)
print("corpus length:", corpus_ids.numel(), "tokens; number of questions:", len(questions))
print("questions and answers:", [" ".join(q) + " -> " + a for q, a in questions])


In [ ]:
# Freeze real weights + teacher target: the correct answer takes 0.85, the rest is split equally
def freeze_except_virtual(model):
    """Freeze all real weights; keep only each layer's virtual keys/values trainable."""
    for p in model.parameters():
        p.requires_grad_(False)
    for name, p in model.named_parameters():
        if "vk" in name or "vv" in name:
            p.requires_grad_(True)


def make_teacher(q, ans):
    """Build a teacher distribution from the question and answer, standing for the answering distribution when the corpus is in context."""
    p = torch.full([V], 0.15 / V)
    p[stoi[ans]] = 0.85
    return p


teacher_targets = {tuple(q): make_teacher(q, ans) for q, ans in questions}
q0, a0 = questions[0]
top = torch.topk(teacher_targets[tuple(q0)], 3)
print(f"teacher distribution (question {' '.join(q0)}, answer {a0}):")
for val, idx in zip(top.values, top.indices):
    print(f"  {vocab[idx]:8s} {val.item():.2f}")


In [ ]:
# Objective 1: NTP cartridge — next-token prediction on the corpus (recitation)
torch.manual_seed(9)
model_ntp = TinyGPT(V)
freeze_except_virtual(model_ntp)
opt = torch.optim.Adam([p for p in model_ntp.parameters() if p.requires_grad],
                       lr=0.05)
losses = []
for step in range(400):
    opt.zero_grad()
    logits = model_ntp(corpus_ids[:, :-1], return_all=True)
    loss = F.cross_entropy(logits.reshape(-1, V),
                           corpus_ids[:, 1:].reshape(-1))
    loss.backward()
    opt.step()
    losses.append(loss.item())
print("NTP cartridge last-step loss:", round(losses[-1], 3),
      "(the goal is to press corpus continuation into the representation, not to answer questions)")


In [ ]:
# Objective 2: context-distillation — the student copies the teacher distribution
torch.manual_seed(9)
model_distill = TinyGPT(V)
freeze_except_virtual(model_distill)
opt = torch.optim.Adam(
    [p for p in model_distill.parameters() if p.requires_grad], lr=0.05)
kls = []
for step in range(500):
    opt.zero_grad()
    loss = 0.0
    for q, ans in questions:
        qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
        s_log = torch.log_softmax(model_distill(qids), dim=-1)
        loss = loss + F.kl_div(s_log, teacher_targets[tuple(q)],
                               reduction="sum")
    loss = loss / len(questions)
    loss.backward()
    opt.step()
    kls.append(loss.item())
print("context-distillation last-step KL:", round(max(0.0, kls[-1]), 3))


In [ ]:
# Comparative evaluation: NTP cartridge vs distilled cartridge on questions
def cartridge_prob(model, q, ans):
    """Probability that the cartridge assigns to the correct answer for the question."""
    qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
    with torch.no_grad():
        logits = model(qids)
    return torch.softmax(logits, dim=-1)[0, stoi[ans]].item()


def kl_to_teacher(model, q):
    """KL(teacher || cartridge) between the cartridge distribution and the teacher."""
    qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
    with torch.no_grad():
        s_log = torch.log_softmax(model(qids), dim=-1)
    p = teacher_targets[tuple(q)]
    return max(0.0, (p * (p.log() - s_log)).sum().item())


import matplotlib.pyplot as plt
import numpy as np

labels = [" ".join(q) for q, _ in questions]
p_ntps = [cartridge_prob(model_ntp, q, ans) for q, ans in questions]
p_dls = [cartridge_prob(model_distill, q, ans) for q, ans in questions]
print("question -> answer | NTP P | distill P | NTP KL | distill KL")
for i, (q, ans) in enumerate(questions):
    k_ntp = kl_to_teacher(model_ntp, q)
    k_dl = kl_to_teacher(model_distill, q)
    print(f"{' '.join(q):16s} -> {ans:8s} | {p_ntps[i]:5.2f} | "
          f"{p_dls[i]:9.2f} | {k_ntp:6.2f} | {k_dl:7.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
ax.bar(x - 0.2, p_ntps, 0.4, label="NTP cartridge")
ax.bar(x + 0.2, p_dls, 0.4, label="distill cartridge")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel("P(correct answer)")
ax.set_title("Question answering: NTP vs distillation")
ax.legend()
plt.tight_layout()
plt.show()

print(f"mean correct probability: NTP {np.mean(p_ntps):.2f}, distill {np.mean(p_dls):.2f}")
print(f"mean KL(teacher||student): NTP "
      f"{np.mean([kl_to_teacher(model_ntp, q) for q, _ in questions]):.2f}, "
      f"distill {np.mean([kl_to_teacher(model_distill, q) for q, _ in questions]):.2f}")
print("Key observation: distillation assigns high probability to the correct answer on every question; NTP only works on patterns it has recited.")


## 4. Reusing already-computed context

This section addresses another kind of memory problem: when the same piece of knowledge is used repeatedly, how to skip duplicate computation. In real systems, RAG (retrieval-augmented generation) splits a large document into chunks, and when the user asks different questions, related chunks are concatenated into the input. Computing the whole input from scratch every time is slow, and it directly decides how long the user waits for the first token. This section makes three things precise: which computation can be reused, what reuse costs in quality, and how to trade the two off.

What we want to reuse is the intermediate result of the model "reading a passage". Recomputing the whole input is expensive; caching the intermediate result and reusing it when the same content reappears saves that compute. The cost is a discrepancy between the cache and a true joint computation, so there are three methods: prefix caching, which reuses only a shared prefix; full KV reuse, which reuses everything; and CacheBlend, which recomputes a fraction on demand.

Each of the three has a shortcoming. Prefix caching reuses KV only for the shared-prefix chunk and recomputes everything after the prefix as usual, so the hit rate is limited. Full KV reuse loads every chunk's KV and skips prefill entirely, but it ignores cross-chunk attention, so quality suffers. CacheBlend proposes selective recompute: per layer, recompute KV for only a small fraction of tokens and keep the cache for the rest, taking both reuse speed and full-recompute quality.

The intuition for selective recompute comes from attention sparsity: only a few "boundary-crossing" tokens have KV that deviate most between a solo computation and a joint computation; recomputing those is enough to restore cross-chunk attention. Below we first compute a 4-token attention example by hand, then look at the attention matrices of the three methods.

### Prefill, reuse, and selective recompute

This subsection makes precise what each of the three methods does and where each falls short, then introduces the core intuition of selective recompute.

RAG splits a document into chunks; each chunk is prefilled independently and its K/V is cached. If a user query arrives and the whole input is prefilled again, tens of thousands of tokens of K/V have to be recomputed, and time to first token grows substantially. Reusing the cache is exactly to skip that duplicate work.

The most basic method is prefix caching: reuse K/V only for a shared prefix, and prefill as usual after the prefix. It requires two requests to share the same start, while in RAG different queries often have different text, so the prefix-reuse hit rate is limited.

Full KV reuse loads every chunk's cached K/V and prefills no chunk. The problem is that chunks were prefilled separately: during a solo prefill, a token's intermediate representation has only interacted with its own chunk; during a joint prefill, it interacts through attention with tokens across chunks, so the intermediate representation differs. Cached K/V therefore deviate from true K/V, and attention quality suffers.

The deviation is not uniform. Tokens near a chunk boundary are affected most by cross-chunk attention, so their deviation is largest; tokens far from the boundary are almost unaffected. Selective recompute uses exactly this: judge token by token, recompute only the small fraction with the largest deviation, and keep the cache for the rest. The recompute ratio r sets the quality-speed trade-off; smaller r is faster, and pressing it too low costs quality.

"Token-level" means the unit of judgment is a single token, not an entire chunk. For each token's K/V in the cache, the system independently decides whether to use the cached value or recompute; the recompute candidates are a set, not a contiguous span. Below we compute this trade-off by hand on a small 4-token attention matrix.

In [ ]:
# Hand calculation: token-level similarity-weighted fusion of attention scores
import numpy as np

# 4 context tokens: chunk A = [a1, a2], chunk B = [b1, b2]
# Full recompute: the whole span is prefilling together to obtain the true keys
K_full = np.array([
    [1.0, 0.0, 0.0, 0.0],   # a1
    [0.0, 1.0, 0.0, 0.0],   # a2
    [0.0, 0.0, 1.0, 0.0],   # b1
    [0.0, 0.0, 0.0, 1.0],   # b2
])
# Cached KV: the two chunks are prefilling independently; boundary tokens deviate from the truth
K_cache = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.5, 0.8, 0.2, 0.1],   # a2 deviates under the influence of chunk B
    [0.1, 0.2, 0.9, 0.2],   # b1 deviates under the influence of chunk A
    [0.0, 0.0, 0.0, 1.0],
])
q = np.array([0.7, 0.3, 0.5, 0.2])   # query token, computed live


def attn_scores(q, K):
    """Token-level attention scores: the query's dot product with each key."""
    return q @ K.T


s_full = attn_scores(q, K_full)
s_cache = attn_scores(q, K_cache)
dev = np.linalg.norm(K_cache - K_full, axis=1)
print("full recompute s_full:", np.round(s_full, 2))
print("cache reuse s_cache:", np.round(s_cache, 2))
print("per-token score deviation:", np.round(np.abs(s_cache - s_full), 2))
print("KV deviation    :", np.round(dev, 2))

r = 0.5
k = int(np.ceil(r * len(K_full)))
hkvd = np.argsort(dev)[::-1][:k]
print("HKVD recompute tokens:", sorted(hkvd.tolist()), "(top", int(r * 100), "%)")
K_cb = K_cache.copy()
K_cb[hkvd] = K_full[hkvd]
s_cb = attn_scores(q, K_cb)
print("CacheBlend s_cb :", np.round(s_cb, 2))
print("deviation after fusion:", np.round(np.abs(s_cb - s_full), 2))
print("Key observation: recomputing 2 boundary tokens returns the attention scores to full recompute.")


### Hand calculation: recomputing two boundary tokens restores attention scores

This subsection computes the 4-token example in the previous cell by hand, to verify that "recomputing boundary tokens is enough to restore attention scores".

The toy data in the code is 4 context tokens, chunk A = [a1, a2], chunk B = [b1, b2]. The four keys are taken as four-dimensional basis vectors, a1=[1,0,0,0], a2=[0,1,0,0], b1=[0,0,1,0], b2=[0,0,0,1], and the query is q=[0.7,0.3,0.5,0.2].

Under full recompute, the attention score is the query's dot product with each key. Because the keys are basis vectors, each score equals the corresponding coordinate of q, s_full = [0.7, 0.3, 0.5, 0.2]. Checking one by one: q dotted with a1 is 0.7, with a2 is 0.3, with b1 is 0.5, with b2 is 0.2.

In the cache, a2 and b1 have drifted: a2's key has mixed in coordinates in the direction of chunk B, b1 has mixed in coordinates in the direction of chunk A, and a1 and b2 stay as they were. The dot products of q with the cached keys become s_cache = [0.7, 0.71, 0.62, 0.2], with deviation |s_cache − s_full| = [0, 0.41, 0.12, 0]. Score deviation hints at which tokens' keys should be recomputed, but it depends on a specific query. CacheBlend uses the more stable KV deviation, the Euclidean distance of the K vector itself from the true value.

Compute KV deviation token by token. a2's cached key differs from the true key by [0.5, −0.2, 0.2, 0.1], a distance $\sqrt{0.5^2+0.2^2+0.2^2+0.1^2} \approx 0.58$; b1 differs by [0.1, 0.2, −0.1, 0.2], a distance $\sqrt{0.1^2+0.2^2+0.1^2+0.2^2} \approx 0.32$; a1 and b2 are both 0. The largest deviations are a2 and b1, exactly the two tokens sitting on the chunk boundary. a1 is at the very start and b2 at the very end; a solo prefill and a joint prefill agree, so spending the recompute budget on them has no return.

r=0.5 means recompute the 2 of 4 with the highest deviation, namely a2 and b1. After replacing their keys with the true values, the scores return to s_cb = [0.7, 0.3, 0.5, 0.2], matching full recompute. Selecting tokens by KV deviation is called HKVD in the paper (Highest KV Deviation). The toy shows one fact: cross-chunk attention only needs a few tokens near the boundary to be repaired, and the attention scores return to the full-recompute level.

In [ ]:
# Attention-matrix comparison of the three methods
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
d = 8
n_a, n_b = 3, 3
T = n_a + n_b
Q = rng.normal(size=(T, d))
K_full = rng.normal(size=(T, d))
V_full = rng.normal(size=(T, d))
# The closer a boundary token is to the chunk edge, the larger the cache deviation
dist = np.array([0, 1, 2, 2, 1, 0])
K_cache = K_full + dist[:, None] * rng.normal(size=(T, d))


def attn_matrix(Q, K, V):
    """Single-head attention matrix, softmax(Q K^T / sqrt(d))."""
    s = Q @ K.T / (d ** 0.5)
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


A_full = attn_matrix(Q, K_full, V_full)
# Full reuse: each chunk computes attention on its own; the cross-chunk block is zeroed
A_reuse = np.zeros_like(A_full)
A_reuse[:n_a, :n_a] = attn_matrix(Q[:n_a], K_full[:n_a], V_full[:n_a])
A_reuse[n_a:, n_a:] = attn_matrix(Q[n_a:], K_full[n_a:], V_full[n_a:])
# CacheBlend: recompute the tokens with the highest KV deviation
dev = np.linalg.norm(K_cache - K_full, axis=1)
k = int(np.ceil(0.5 * T))
hkvd = np.argsort(dev)[::-1][:k]
K_cb = K_cache.copy()
K_cb[hkvd] = K_full[hkvd]
A_cb = attn_matrix(Q, K_cb, V_full)


def fro_norm(A, B):
    """Frobenius-norm gap between two attention matrices."""
    return np.linalg.norm(A - B)


print("attention deviation ||A_reuse - A_full||:", round(fro_norm(A_reuse, A_full), 3))
print("attention deviation ||A_cb   - A_full||:", round(fro_norm(A_cb, A_full), 3))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ["full recompute", "full KV reuse", "CacheBlend"]
for ax, A, title in zip(axes, [A_full, A_reuse, A_cb], titles):
    ax.imshow(A, cmap="YlOrRd", vmin=0, vmax=0.6)
    ax.set_title(title)
    ax.set_xlabel("key token")
    ax.set_ylabel("query token")
plt.tight_layout()
plt.show()
print("Darker heatmap cells mean higher attention; the cross-chunk block of full reuse is zeroed.")


In [ ]:
# Sparsity of KV deviation: a few tokens contribute most of the deviation
import numpy as np
import matplotlib.pyplot as plt

dev_sorted = np.sort(dev)[::-1]
cdf = np.cumsum(dev_sorted) / dev_sorted.sum()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, T + 1), cdf, marker="o")
ax.axhline(0.9, color="gray", linestyle="--", label="90% deviation")
ax.set_xlabel("top-k tokens")
ax.set_ylabel("cumulative deviation fraction")
ax.set_title("KV deviation is concentrated")
ax.legend()
plt.tight_layout()
plt.show()
top_k = int(np.flatnonzero(cdf >= 0.9)[0]) + 1
print(f"the top {top_k} tokens contribute 90% of the deviation")

# Recompute the same k tokens: HKVD vs picking at random (low-deviation tokens), which recovers better
rand = np.argsort(dev)[:k]      # deliberately pick the k with the smallest deviation


def recovered(K):
    """Attention deviation from full recompute, using the given key matrix."""
    A = attn_matrix(Q, K, V_full)
    return np.linalg.norm(A - A_full)


K_hkvd = K_cache.copy()
K_hkvd[hkvd] = K_full[hkvd]
K_rand = K_cache.copy()
K_rand[rand] = K_full[rand]
print("HKVD recompute attention deviation:", round(recovered(K_hkvd), 3))
print("low-deviation token recompute deviation:", round(recovered(K_rand), 3))
print("Key observation: with the same recompute budget, HKVD recovers close to full recompute; arbitrary selection does not.")


In [ ]:
# Progressive filtering: adjacent-layer KV deviation is highly correlated, so a full pass on every layer is unnecessary
import numpy as np
from scipy.stats import spearmanr


def top_r_percent(dev, r):
    """Return token indices in the top r fraction of KV deviation."""
    k = max(1, int(np.ceil(r * len(dev))))
    return np.argsort(dev)[::-1][:k]


rng = np.random.default_rng(7)
base_dev = rng.uniform(0.1, 1.0, size=12)
devs = [np.clip(base_dev + rng.normal(scale=0.05 * (l + 1), size=12),
                0, None) for l in range(4)]

ratios = [0.30, 0.20, 0.15, 0.10]
cand = None
for l, (dev, r) in enumerate(zip(devs, ratios)):
    if cand is None:
        cand = top_r_percent(dev, r)
    else:
        cand = cand[top_r_percent(dev[cand], r / ratios[l - 1])]
    print(f"layer {l}: {len(cand)} candidates")

for l in range(len(devs) - 1):
    rho, _ = spearmanr(devs[l], devs[l + 1])
    print(f"layer {l} vs {l + 1} Spearman rank correlation = {rho:.3f}")
print("Key observation: adjacent-layer deviation is highly correlated, so we can filter inside the candidate set layer by layer.")


## 5. Engineering a memory system

This section puts the three lines together, and looks at how a real memory system is built in practice. We assign the three layers of work, then look at a cost-saving detail of CacheBlend: how to make recompute take almost no extra time.

Read together, the three papers split a memory system's work into three layers: MemGPT decides which storage level memory lives on, Cartridges decide what representation memory takes, and CacheBlend lets the same memory be reused without paying prefill again.

CacheBlend's systems engineering pipelines selective recompute with loading KV from slow storage: while recomputing layer i, the next layer i+1 is loaded in the background. As long as recompute time does not exceed load time, recompute is free, so the KV cache can sit on slower, cheaper devices without increasing time to first token. Below we reproduce this decision with a latency model.

In [ ]:
# Latency model: pipelining recompute with KV load
import numpy as np
import matplotlib.pyplot as plt

# Per-layer KV cache size (Llama-7B scale, 2048-token context, fp16)
kv_bytes = 2048 * 2 * 4096 * 2          # token × (K+V) × dim × 2 bytes
throughput = {"HBM": 4e12, "NVMe SSD": 2e9, "SATA SSD": 5.5e8, "HDD": 1.5e8}


def T_load(device):
    """Time to load one layer of KV from device (milliseconds)."""
    return kv_bytes / throughput[device] * 1000


def T_recompute(r, prefill_layer_ms=20):
    """Time to recompute an r fraction of tokens for one layer (milliseconds)."""
    return r * prefill_layer_ms


print("one-layer KV cache:", round(kv_bytes / 1e6, 1), "MB")
print("recompute 15% of tokens, one layer:", f"{T_recompute(0.15):.1f} ms")
for dev in throughput:
    t = T_load(dev)
    hidden = "recompute hidden by load latency" if t >= T_recompute(0.15) else "recompute is the bottleneck"
    print(f"{dev:8s} load one layer {t:7.1f} ms  | {hidden}")

# Pipeline latency = max(recompute, load)
rs = np.linspace(0, 1, 100)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rs, np.maximum(T_recompute(rs), T_load("NVMe SSD")), label="NVMe SSD")
ax.plot(rs, np.maximum(T_recompute(rs), T_load("HBM")), label="HBM")
ax.axvline(0.15, color="gray", linestyle="--", label="quality floor 15%")
ax.set_xlabel("recompute ratio r")
ax.set_ylabel("per-layer latency (ms)")
ax.set_title("Pipelined recompute vs KV load")
ax.legend()
plt.tight_layout()
plt.show()
print("The NVMe curve stays flat for r < 0.84: load latency covers recompute latency.")


### Why recompute can be free

This subsection explains the key numbers in the previous latency model: why recompute can be free, and how the pipeline is arranged.

The latency model above shows a systems-engineering detail of CacheBlend: pipeline recompute with load. While recomputing the selected tokens of layer i, layer i+1's KV is loaded from slow storage in the background. The two operations run in parallel; as long as recompute time does not exceed load time, recompute is hidden inside the load, and per-layer total latency is the larger of the two rather than their sum.

The volume of one layer's KV cache is determined: Llama-7B scale, 2048 tokens, K and V each 4096-dimensional, 2 bytes per dimension, i.e. $2048 \times 2 \times 4096 \times 2 \approx 33.6$ MB. Load time depends on storage speed: HBM about 0.008 ms, NVMe SSD about 16.8 ms, SATA SSD about 61 ms, HDD about 224 ms. Recomputing 15% of tokens for one layer takes about 3 ms.

Compare which hides which. NVMe load 16.8 ms is greater than recompute 3 ms, so total latency takes the load value and recompute is free; SATA and HDD even more so. HBM load is only 0.008 ms, far smaller than recompute, so recompute becomes the bottleneck, but a cache already on HBM has no load problem. In the plot the NVMe curve stays flat for r < 0.84 for this reason: as long as recompute does not exceed 16.8 ms (i.e. r ≤ 0.84), latency is determined by load.

The value of this design is that the KV cache can sit on cheaper, slower devices without increasing time to first token. Recompute runs on the GPU, load travels the storage bus, the two are independent, and after overlap the total time equals the slower side. The quality floor keeps r at 15% or above; in the interval from 0.15 to 0.84, recompute on NVMe is completely invisible to latency.

## Summary

This lecture completed a three-layer memory implementation:

- [ ] Why memory matters: the context window is a hard cap, eviction policy decides yield order, and FIFO / importance / LRU each have trade-offs
- [ ] MemGPT: two-level main context and external context storage; a function mechanism lets the model read and write memory itself; memory pressure triggers eviction and recursive summaries
- [ ] Eviction and budget: a 70% warning line and a 100% flush line; after eviction the main context returns within budget; evicted facts can be recalled via recall
- [ ] Cartridges: distill a corpus into trainable key/value vectors; context-distillation lets the student copy the output distribution of "corpus in context"
- [ ] CacheBlend: reuse precomputed KV across chunks; selectively recomputing high-deviation tokens restores cross-chunk attention
- [ ] Engineering practice: overlapping recompute with KV load in a pipeline, so even slow storage can hide recompute latency

The next main thread is evaluation: whether an Agent's memory and tools are doing the right thing needs an evaluation framework that can run for a long time and can tell true progress from false performance.


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

All three short problems are based on code written in this lecture. Complete each one on paper first, then run and compare.


**Exercise 1: eviction and recursive summary on a MemGPT queue**

Fill in two places in the functions below: `should_flush` judges whether the main context has reached the flush line; `make_summary` merges the old summary with the evicted message into a recursive summary. The assertions check the flush-line judgment and the budget after eviction.

Hint: eviction takes the oldest message in the queue; a recursive summary has to take both the old summary and the evicted message into account.


In [ ]:
# Blank 1: judge whether the flush line has been reached
def should_flush(tokens, budget):
    """Whether the main-context token count has reached the flush line."""
    return tokens >= budget


# Blank 2: recursive summary = old summary + evicted message
def make_summary(client, old, evicted):
    """Merge the old summary and the evicted message into a recursive summary."""
    if False:
        return old + " ~ " + evicted
    prompt = f"Old summary: {old}\nEvicted message: {evicted}\nMerge into a new summary:"
    return client.chat([{"role": "user", "content": prompt}])


budget = 60
msgs = ["m" * 26, "n" * 26, "p" * 26]          # 26 characters each
total = sum(len(m) for m in msgs)
assert should_flush(total, budget) is True, "78 >= 60 should flush"
left = total - len(msgs[0])
assert should_flush(left, budget) is False, "after evicting one item, occupancy should be back within budget"
summary = make_summary(client, "(empty)", msgs[0])
assert len(summary) > 0, "the recursive summary must be nonempty"
print("Exercise 1 passed: can judge the flush line, occupancy returns within budget after eviction, and a recursive summary has been produced.")


**Exercise 2: HKVD selection and inter-layer correlation**

Complete `top_r_percent` to select the top r fraction of tokens with the highest KV deviation; complete `layer_correlation` to measure agreement of adjacent-layer deviation with Spearman rank correlation. The assertions check the selected set and the correlation threshold.

Hint: `np.argsort(dev)[::-1]` gives descending indices; `spearmanr` returns the pair (rho, pvalue).


In [ ]:
import numpy as np
from scipy.stats import spearmanr


def top_r_percent(dev, r):
    """Return token indices in the top r fraction of KV deviation."""
    k = max(1, int(np.ceil(r * len(dev))))
    return np.argsort(dev)[::-1][:k]


def layer_correlation(dev_a, dev_b):
    """Spearman rank correlation of two layers' KV deviation."""
    rho, _ = spearmanr(dev_a, dev_b)
    return rho


dev_l0 = np.array([0.1, 0.9, 0.2, 0.8, 0.15, 0.7])
dev_l1 = np.array([0.12, 0.85, 0.18, 0.82, 0.13, 0.72])
assert set(top_r_percent(dev_l0, 0.5)) == {1, 3, 5}, "should select the 3 tokens with largest deviation"
assert layer_correlation(dev_l0, dev_l1) > 0.9, "adjacent-layer deviation should be highly correlated"
print("Exercise 2 passed: HKVD selected the top 50% of tokens, and inter-layer Spearman correlation is above 0.9.")


**Exercise 3: the KL objective of context-distillation**

Complete `distillation_loss`: turn teacher logits and student log-logits into distributions, then compute KL(teacher || student). The assertions check that a student aligned with the teacher has lower loss than a uniform student.

Hint: the teacher uses softmax to become a probability; the student uses log_softmax to become log probability; KL = sum p (log p - log q).


In [ ]:
import torch


def distillation_loss(teacher_logits, student_logits):
    """KL(teacher || student); the teacher comes from corpus-in-context, the student from the cartridge."""
    p = torch.softmax(teacher_logits, dim=-1)
    log_q = torch.log_softmax(student_logits, dim=-1)
    return (p * (p.log() - log_q)).sum(dim=-1).mean()


teacher_logits = torch.tensor([[0.1, 0.1, 2.0, 0.2, 0.1]])
student_uniform = torch.tensor([[0.3, 0.3, 0.3, 0.3, 0.3]])
student_aligned = torch.tensor([[0.1, 0.1, 2.0, 0.2, 0.1]])
loss_bad = distillation_loss(teacher_logits, student_uniform)
loss_good = distillation_loss(teacher_logits, student_aligned)
assert loss_good.item() < loss_bad.item(), "an aligned distribution should make KL smaller"
print("Exercise 3 passed: the KL objective aligns the cartridge with the teacher's answering distribution.")


## References

- Packer et al., [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560), 2023 — OS-style hierarchical memory: two-level main/external context storage and model-directed function reads and writes
- Berglund et al., [Cartridges: Lightweight and general-purpose long context representations via self-study](https://arxiv.org/abs/2506.06266), 2025 — distill a corpus offline into a trainable KV representation, with self-study synthetic data and a context-distillation objective
- Huang et al., [CacheBlend: Fast LLM Serving for RAG with Cached Knowledge Fusion](https://arxiv.org/abs/2405.16444), 2024 — reuse precomputed KV across chunks; selectively recompute high-deviation tokens to restore cross-chunk attention
- Liu et al., [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172), 2023 — empirical evidence of lost middle information in long-context models, a motivation for MemGPT
- Li & Liang, [Prefix-Tuning: Optimizing Continuous Prompts for Generation](https://arxiv.org/abs/2101.00190), 2021 — the original work on trainable prefix vectors, the source of Cartridge parameterization
- Chen et al., [PromptCache: Modular Attention Reuse for Low-latency Inference](https://arxiv.org/abs/2311.04934), 2023 — the full KV-reuse baseline, the variant that ignores cross-chunk attention
- [Letta](https://github.com/letta-ai/letta) — the open-source engineered implementation of MemGPT, a long-term memory agent framework
- [CS329A course homepage](https://cs329a.stanford.edu/) — Autumn 2025 syllabus; this lecture sits in the unit on adding memory to Agents
